In [ ]:
import os
import sys
if 'google.colab' in sys.modules:
  !pip install -q numba imageio
  !apt-get install -y -qq ffmpeg

os.makedirs('../results/gifs', exist_ok=True)
os.makedirs('../results/plots', exist_ok=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

In [ ]:
L, N, dt = 2, 100, 1
Du, Dv = 2e-5, 1e-5
snapshots = [0, 100, 250, 500, 1000, 1500, 2500, 3500, 5000, 7000, 10000]

F, k = 0.025, 0.055

In [ ]:
def laplacian(array, mode):
    if mode == 1:
        return (np.roll(array, 1) + np.roll(array, -1) - 2 * array) / ((L / N) ** 2)
    else:
        return (np.roll(array, 1, axis=0) + np.roll(array, -1, axis=0) +
                np.roll(array, 1, axis=1) + np.roll(array, -1, axis=1) - 4 * array) / ((L / N) ** 2)

def step(u, v, F, k, mode):
    u_old = u
    v_old = v

    u_lap = laplacian(u, mode)
    v_lap = laplacian(v, mode)

    uvv = u_old * (v_old ** 2)
    dudt = Du * u_lap - uvv + F - F * u_old
    dvdt = Dv * v_lap + uvv - (F + k) * v_old

    u_new = u_old + dt * dudt
    v_new = v_old + dt * dvdt

    return u_new, v_new

def get_final_state(F_val, k_val, steps, mode):
    u = np.ones((N,N))
    v = np.zeros((N,N))
    
    N1, N2 = N // 4, 3 * N // 4
    u[N1:N2, N1:N2] = 0.5 + np.random.rand(N2-N1, N2-N1) * 0.1
    v[N1:N2, N1:N2] = 0.25 + np.random.rand(N2-N1, N2-N1) * 0.1
    
    for _ in range(steps):
        u, v = step(u, v, F_val, k_val, mode)
        
    return u

def run_simulation(u, v, F, k, mode, snapshots):
    u_history = []
    v_history = []
    for i in range(snapshots[-1] + 1):
        u, v = step(u, v, F, k, mode)

        if i in snapshots:
            u_history.append(u)
            v_history.append(v)

    return u_history, v_history

In [ ]:
# Zadanie 1
u = np.ones(N)
v = np.zeros(N)
mode = np.shape(np.shape(u))[0]
N1, N2 = N // 4, 3 * N // 4

u[N1:N2+1] = 0.4 + np.random.rand(N2-N1+1) * 0.2
v[N1:N2+1] = 0.2 + np.random.rand(N2-N1+1) * 0.2

In [ ]:
# Zadanie 2
u = np.ones((N, N))
v = np.zeros((N, N))
mode = np.shape(np.shape(u))[0]
N1, N2 = N // 4, 3 * N // 4

u[N1:N2+1, N1:N2+1] = 0.4 + np.random.rand(N2-N1+1, N2-N1+1) * 0.2
v[N1:N2+1, N1:N2+1] = 0.2 + np.random.rand(N2-N1+1, N2-N1+1) * 0.2

# # Jak wygląda ewolucja układu dla różnych wartości F i k, np.
# Fs = [.025, .03, .01, .04, .052, .037]
# ks = [.055, .062, .047, .062, .0615, .06]

In [ ]:
u_history, v_history = run_simulation(u, v, F, k, mode, np.arange(0, 10000, 100, dtype='int'))

fig, ax = plt.subplots(figsize=(8, 8))
img = ax.imshow(u_history, interpolation='nearest')
img.set_clim(vmin=0, vmax=1)

cbar = fig.colorbar(img, ticks=[0, 0.3, 0.5, 1], orientation='vertical')
title = ax.set_title(f"1D Gray-Scott Space-time")
ax.axis('off')

In [ ]:
u_history, v_history = run_simulation(u, v, F, k, mode, snapshots)


if (np.shape(np.shape(u))[0] == 1):
    fig, ax = plt.subplots(figsize=(8, 5))

    line_u, = ax.plot(u_history[0], label='u', color='blue')
    line_v, = ax.plot(v_history[0], label='v', color='red')

    ax.set_ylim(0, 1)
    ax.set_xlim(0, N)
    ax.legend('upper right')
    title = ax.set_title(f"Gray-Scott 1D (Step: 0)")

    def update(frame_idx):
        line_u.set_ydata(u_history[frame_idx])
        line_v.set_ydata(v_history[frame_idx])
        
        frame_nr = snapshots[frame_idx]
        title.set_text(f"Step: {frame_nr}")
        return [line_u, line_v, title]
    
else:
    fig, ax = plt.subplots(figsize=(8, 8))
    img = ax.imshow(u_history[0], interpolation='nearest')
    img.set_clim(vmin=0, vmax=1)

    cbar = fig.colorbar(img, ticks=[0, 0.3, 0.5, 1], orientation='vertical')
    title = ax.set_title(f"Gray-Scott Reaction (Step: 0)")
    ax.axis('off')

    def update(frame_idx):
        img.set_data(u_history[frame_idx])
        frame_nr = snapshots[frame_idx]
        title.set_text(f"Step: {frame_nr}")
        return [img, title]

anim = animation.FuncAnimation(
    fig,
    update,
    frames=len(u_history),
    interval=250,
    blit=True
)

plt.show()
anim.save(f"../results/gifs/11_gray_scott_{np.shape(np.shape(u))[0]}D.gif", writer='pillow', fps=4)

In [ ]:
k_value = 0.062
F_values = np.linspace(0.02, 0.10, 81)

frames = []

for idx, f_val in enumerate(F_values):
    final_u = get_final_state(f_val, k_value, 4000, 2)
    frames.append(final_u)

fig, ax = plt.subplots(figsize=(7, 7))

img = ax.imshow(frames[0], cmap='viridis', interpolation='bicubic', vmin=0, vmax=1)
title = ax.set_title(f"Phase Diagram Slice\nk={k_value}, F={F_values[0]:.4f}")
ax.axis('off')

text_template = 'k = {:.3f}\nF = {:.4f}'
text = ax.text(0.05, 0.95, '', transform=ax.transAxes, color='white', fontsize=12, va='top')

def update(frame_idx):
    current_u = frames[frame_idx]
    current_F = F_values[frame_idx]
    
    img.set_data(current_u)
    title.set_text(f"Phase Diagram Slice (k={k_value})")
    text.set_text(text_template.format(k_value, current_F))
    
    return [img, title, text]

anim = animation.FuncAnimation(
    fig,
    update,
    frames=len(frames),
    interval=250,
    blit=True
)

plt.show()
anim.save("../results/gifs/11_gray_scott_phase_diagram.gif", writer='pillow', fps=4)